 # 03b_RLVR_Manual_Analysis_v18.ipynb



 ### 🔬 Granular Out-of-Sample Performance & Factor Allocation Inspection

 - **Observation Space:** 46 Dimensions (12 CS Mean, 12 CS Std, 12 Benchmark Vector, 10 Stationary Macro)

 - **Action Space:** 16 Dimensions (12 Feature Weights, Offset, Width, Equity Exposure, Active Tilt)

 - **Analysis:** Training diagnostics, factor tilts, trade execution blotters, universe selection, and crash analysis.

In [ ]:
# CELL 1: Environment Setup & Automatic Top-Model Checkpoint Finder
import gc
import os
import pickle
import re
import sys
from pathlib import Path
from typing import Any, Dict

from IPython.display import display
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.settings import CacheConfig, TradingConfig
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.validator import AgentEvaluator
from strategy.registry import get_strategy_registry

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"

if not checkpoint_dir.exists():
    raise FileNotFoundError(f"❌ Checkpoint directory does not exist: {checkpoint_dir}")

# Discover best model checkpoints sorted by rolling and validation Sharpe
pattern = re.compile(
    r"^(model_.*?)(?:_ep_\d+)?(?:_rollsh_([-0-9.]+))?_valsh_([-0-9.]+)\.pt$"
)
best_models_per_set = {}

for file_path in sorted(checkpoint_dir.glob("*.pt")):
    match = pattern.match(file_path.name)
    if match:
        param_key, rollsh_str, valsh_str = match.groups()
        rollsh = float(rollsh_str) if rollsh_str is not None else -float("inf")
        valsh = float(valsh_str) if valsh_str is not None else -float("inf")
        score_tuple = (rollsh, valsh)

        if (
            param_key not in best_models_per_set
            or score_tuple > best_models_per_set[param_key][1]
        ):
            best_models_per_set[param_key] = (file_path.name, score_tuple)

if not best_models_per_set:
    raise RuntimeError(f"❌ No valid checkpoint `.pt` files found in: {checkpoint_dir}")

print("🔍 Top Saved Model Checkpoints:")
for i, (fname, (rollsh, valsh)) in enumerate(best_models_per_set.values(), 1):
    print(f'  #{i:02d} [rollsh={rollsh:.2f}, valsh={valsh:.2f}] -> "{fname}"')

# Default target_file set to highest scoring model
default_target_file = list(best_models_per_set.values())[0][0]
target_file = default_target_file
print(f"\n👉 Active target_file set to: {target_file}")

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output

🔍 Top Saved Model Checkpoints:
  #01 [rollsh=1.06, valsh=0.98] -> "model_T1_Convex_25x_HighTilt85_s42_ent0.0035_mult2.5_tilt0.85_w3_g0.9_rsc20.0_lr0.00015_ep_64_rollsh_1.06_valsh_0.98.pt"
  #02 [rollsh=1.03, valsh=1.10] -> "model_T1_Convex_2x_Scale20_s42_ent0.0035_mult2.0_tilt0.7_w3_g0.9_rsc20.0_lr0.00015_ep_80_rollsh_1.03_valsh_1.10.pt"
  #03 [rollsh=0.71, valsh=0.60] -> "model_T1_Convex_3x_Scale20_s42_ent0.0035_mult3.0_tilt0.7_w3_g0.9_rsc20.0_lr0.00015_ep_10_rollsh_0.71_valsh_0.60.pt"
  #04 [rollsh=0.79, valsh=0.93] -> "model_T

In [ ]:
# CELL 2: Manual Target File Override (Set specific file or run Cell 1 default)
# To inspect a specific checkpoint, uncomment and update the string below:
target_file = "model_T2_LR20_Ent005_Scale25_s42_ent0.005_mult2.5_tilt0.75_w3_g0.9_rsc25.0_lr0.0002_ep_72_rollsh_1.14_valsh_1.12.pt"

In [ ]:
# CELL 3: Checkpoint Loading & Full OOS Action/State Reconstruction
if isinstance(target_file, (tuple, list)):
    target_file = target_file[0]
target_file = str(target_file).strip().replace('"', "")

target_path = checkpoint_dir / target_file
if not target_path.exists():
    target_path = OUTPUT_DIR / target_file

if not target_path.exists():
    raise FileNotFoundError(f"❌ Checkpoint file not found: {target_file}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🤖 Loading PyTorch Checkpoint: {target_path.name}")

checkpoint = torch.load(target_path, map_location=device, weights_only=False)

grid_params = {}
if isinstance(checkpoint, dict):
    grid_params = checkpoint.get("grid_params", checkpoint.get("metadata", {}))
    model_state = checkpoint.get("model_state_dict", checkpoint)
    scaler_state = checkpoint.get("scaler_state", None)
else:
    model_state = checkpoint
    scaler_state = None

# Dynamically reconstruct TradingConfig
config = TradingConfig()

lossav_match = re.search(r"_(?:lossav|pen)_([0-9.]+)_", target_file)
cash_match = re.search(r"_cash_([0-9.]+)_", target_file)
tilt_match = re.search(r"_(?:tilt|atilt)_([0-9.]+)_", target_file)
w_match = re.search(r"_w([0-9]+)_", target_file)
mult_match = re.search(r"_mult([0-9.]+)_", target_file)

if "loss_aversion_penalty" in grid_params:
    config.loss_aversion_penalty = float(grid_params["loss_aversion_penalty"])
elif lossav_match:
    config.loss_aversion_penalty = float(lossav_match.group(1))

if "max_cash_pct" in grid_params:
    config.max_cash_pct = float(grid_params["max_cash_pct"])
elif cash_match:
    config.max_cash_pct = float(cash_match.group(1))

if "min_active_tilt" in grid_params:
    config.min_active_tilt = float(grid_params["min_active_tilt"])
elif tilt_match:
    config.min_active_tilt = float(tilt_match.group(1))

if "min_basket_width" in grid_params:
    config.min_basket_width = int(grid_params["min_basket_width"])
elif "min_width" in grid_params:
    config.min_basket_width = int(grid_params["min_width"])
elif w_match:
    config.min_basket_width = int(w_match.group(1))

if "UPSIDE_ALPHA_MULT" in grid_params:
    config.upside_alpha_mult = float(grid_params["UPSIDE_ALPHA_MULT"])
elif mult_match:
    config.upside_alpha_mult = float(mult_match.group(1))

print(
    f"🔧 Reconstructed TradingConfig: loss_aversion = {config.loss_aversion_penalty}, "
    f"max_cash = {config.max_cash_pct}, min_active_tilt = {config.min_active_tilt}, "
    f"min_basket_width = {config.min_basket_width}, upside_mult = {config.upside_alpha_mult}"
)

print("   -> Initializing Environments for OOS Action Reconstruction...")
df_ohlcv, macro_df, features_df = load_processed_data()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
feature_cube = pd.read_parquet(cache_file_path)

# Forward 1-Day MTM Simple Return Matrix (P_{t+1} - P_t) / P_t
reward_matrix = df_close.pct_change(1, fill_method=None).shift(-1)
reward_matrix["CASH"] = 0.0

valid_dates = feature_cube.index.get_level_values("Date").unique()
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]
TEST_START = pd.Timestamp("2022-04-01")
cal_test = valid_calendar[valid_calendar >= TEST_START]

env_test = DiscoveryEnv(
    feature_cube=feature_cube,
    simple_ret_matrix=reward_matrix,
    calendar=cal_test,
    macro_df=macro_df,
    config=config,
)
gym_test = RLVRGymEnv(env_test, macro_df)
gym_test.is_training = False

obs_dim = gym_test.observation_space.shape[0]
action_dim = gym_test.action_space.shape[0]
assert obs_dim == 46, f"Expected 46 observation dimensions, got {obs_dim}"
assert action_dim == 16, f"Expected 16 action dimensions, got {action_dim}"

agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)
agent.load_state_dict(model_state)

if scaler_state is not None:
    gym_test.scaler.load_state(scaler_state)
    print("   -> [SCALER SYNC] Applied saved observation scaler state.")

print("   -> Executing Deterministic OOS Evaluation...")
results = AgentEvaluator.evaluate(
    agent=agent, env=gym_test, device=device, detailed_log=True
)
if isinstance(checkpoint, dict):
    results["metadata"] = grid_params
    results["training_history"] = checkpoint.get("training_history", [])

target_stem = Path(target_file).stem
results_save_path = OUTPUT_DIR / f"results_{target_stem}.pkl"
with open(results_save_path, "wb") as f:
    pickle.dump(results, f)

blotter_df = pd.DataFrame(results.get("blotter", []))
if not blotter_df.empty:
    date_col = (
        blotter_df["date"] if "date" in blotter_df else blotter_df["decision_date"]
    )
    blotter_df["date"] = pd.to_datetime(date_col)
    blotter_df = blotter_df.sort_values("date").reset_index(drop=True)
    blotter_path = OUTPUT_DIR / f"blotter_df_{target_stem}.parquet"
    blotter_df.to_parquet(blotter_path)
    print(f"✅ Saved OOS blotter to: {blotter_path}")

print(f"✅ Total OOS days evaluated: {len(blotter_df)}")

🤖 Loading PyTorch Checkpoint: model_T2_LR20_Ent005_Scale25_s42_ent0.005_mult2.5_tilt0.75_w3_g0.9_rsc25.0_lr0.0002_ep_72_rollsh_1.14_valsh_1.12.pt
🔧 Reconstructed TradingConfig: loss_aversion = 0.0, max_cash = 0.0, min_active_tilt = 0.75, min_basket_width = 3, upside_mult = 2.5
   -> Initializing Environments for OOS Action Reconstruction...
   -> [SCALER SYNC] Applied saved observation scaler state.
   -> Executing Deterministic OOS Evaluation...
✅ Saved OOS blotter to: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output\blotter_df_model_T2_LR20_Ent005_Scale25_s42_ent0.005_mult2.5_tilt0.75_w3_g0.9_rsc25.0_lr0.0002_ep_72_rollsh_1.14_valsh_1.12.parquet
✅ Total OOS days evaluated: 1097


In [ ]:
# CELL 4: Training Diagnostics (Metadata & 2x3 Subplot Grid)
training_history = results.get("training_history", [])
metadata = results.get("metadata", {})

if training_history:
    history_df = pd.DataFrame(training_history)

    if metadata:
        print("=" * 50)
        print("⚙️ HYPERPARAMETERS USED FOR THIS RUN:")
        for k, v in metadata.items():
            print(f" - {str(k).ljust(25)} : {v}")
        print("=" * 50)

    epochs = history_df["epoch"]

    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=(
            "Total & Policy Loss",
            "Value Loss (Critic)",
            "Entropy (Exploration)",
            "PPO Stability (Approx KL & Clip Frac)",
            "Explained Variance (Value Accuracy)",
            "Average Step Reward",
        ),
        shared_xaxes=True,
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    # (1, 1): Losses
    fig.add_trace(
        go.Scatter(
            x=epochs, y=history_df.get("total_loss"), mode="lines", name="Total Loss"
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=epochs, y=history_df.get("policy_loss"), mode="lines", name="Policy Loss"
        ),
        row=1,
        col=1,
    )

    # (1, 2): Value Loss
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=history_df.get("value_loss"),
            mode="lines",
            name="Value Loss",
            line=dict(color="orange"),
        ),
        row=1,
        col=2,
    )

    # (1, 3): Entropy
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=history_df.get("entropy"),
            mode="lines",
            name="Entropy",
            line=dict(color="green"),
        ),
        row=1,
        col=3,
    )

    # (2, 1): Stability
    if "approx_kl" in history_df.columns:
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=history_df["approx_kl"],
                mode="lines",
                name="Approx KL",
                line=dict(color="teal"),
            ),
            row=2,
            col=1,
        )
    if "clip_fraction" in history_df.columns:
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=history_df["clip_fraction"],
                mode="lines",
                name="Clip Fraction",
                line=dict(color="crimson", dash="dot"),
            ),
            row=2,
            col=1,
        )

    # (2, 2): Explained Variance
    if "explained_variance" in history_df.columns:
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=history_df["explained_variance"],
                mode="lines",
                name="Explained Var",
                line=dict(color="brown"),
            ),
            row=2,
            col=2,
        )
        fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=2)
        fig.add_hline(y=0.0, line_dash="dot", line_color="red", row=2, col=2)

    # (2, 3): Step Reward
    fig.add_trace(
        go.Scatter(
            x=epochs,
            y=history_df.get("avg_reward"),
            mode="lines",
            name="Avg Reward",
            line=dict(color="purple"),
        ),
        row=2,
        col=3,
    )

    fig.update_layout(
        height=750,
        width=1200,
        title_text=f"🧠 Agent Training Diagnostics ({target_file})",
    )
    fig.show()

    print("\nLast 5 Training Epochs:")
    print(history_df.tail())
else:
    print("⚠️ No training history found in results checkpoint.")

⚙️ HYPERPARAMETERS USED FOR THIS RUN:
 - ID                        : T2_LR20_Ent005_Scale25
 - LR                        : 0.0002
 - CRITIC_LR                 : 0.0008
 - ENT_COEF_START            : 0.005
 - ENT_COEF_END              : 0.0005
 - UPSIDE_ALPHA_MULT         : 2.5
 - loss_aversion_penalty     : 0.0
 - max_cash_pct              : 0.0
 - min_active_tilt           : 0.75
 - min_width                 : 3
 - GAMMA                     : 0.9
 - GAE_LAMBDA                : 0.95
 - REWARD_SCALE              : 25.0
 - SEEDS                     : [42]
 - NUM_EPOCHS                : 80
 - UPDATE_EPOCHS             : 4
 - PATIENCE                  : 18
 - NUM_STEPS                 : 1024
 - MINI_BATCH_SIZE           : 256
 - NUM_ENVS                  : 8
 - EVAL_FREQ                 : 2
 - ROLLING_EVAL_WINDOW       : 3
 - WARMUP_EPOCHS             : 8



Last 5 Training Epochs:
    policy_loss  value_loss    entropy  total_loss  approx_kl  clip_fraction  \
67    -0.013341    0.221885  14.857842    0.079308   0.004660       0.053131   
68    -0.011529    0.233007  14.857888    0.087516   0.004042       0.041382   
69    -0.013570    0.144810  14.858127    0.042212   0.004379       0.050049   
70    -0.011341    0.204189  14.855153    0.074970   0.003958       0.041626   
71    -0.008934    0.331785  14.851383    0.142014   0.003102       0.026794   

    explained_variance  epoch  avg_reward  
67            0.762790     68    0.004556  
68            0.710569     69    0.004892  
69            0.783610     70    0.004013  
70            0.754928     71    0.004123  
71            0.748283     72    0.004863  


In [ ]:
# CELL 5: Action Space Mapping (16 Dimensions)
registry = get_strategy_registry(config)

action_names = list(registry.keys()) + [
    "Rank Offset",
    "Rank Width",
    "Equity Exposure",
    "Active Tilt",
]

actions = np.vstack(blotter_df["raw_actions"].values)
assert actions.shape[1] == 16, f"Expected 16 actions, got {actions.shape[1]}"

for i, name in enumerate(action_names):
    blotter_df[name] = actions[:, i]

print("=== Action Space Mapping (16 Dimensions) ===")
for i, name in enumerate(action_names):
    print(f"  [{i:02d}] {name}")

=== Action Space Mapping (16 Dimensions) ===
  [00] Log Price Gain
  [01] Sharpe (TRP)
  [02] Momentum (21d)
  [03] Info Ratio (63d)
  [04] Oversold (-RSI)
  [05] Dip Buyer (-dd_21)
  [06] Range Position (20d)
  [07] Return Autocorr (15d)
  [08] Low Volatility (-ATRP)
  [09] Slope_P_5_Z
  [10] Slope_V_5_Z
  [11] Convexity
  [12] Rank Offset
  [13] Rank Width
  [14] Equity Exposure
  [15] Active Tilt


In [ ]:
# CELL 6: Observation Space Mapping (46 Dimensions)
cross_sectional_features = [
    f"{name} ({'Z' if bp.scaling_type == 'Z-Score' else 'S'})"
    for name, bp in registry.items()
]
macro_features = [
    "Mkt_Ret",
    "Mkt_Ret_Z",
    "Macro_Trend",
    "Macro_Trend_Z",
    "Yield_Curve_10Y2Y_Z",
    "Macro_Trend_Vel_Z",
    "Macro_Trend_Mom",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "Mkt_Vol_63d_Z",
]
benchmark_name = config.benchmark_ticker

obs_names = (
    [f"Mean: {f}" for f in cross_sectional_features]
    + [f"Std: {f}" for f in cross_sectional_features]
    + [f"{benchmark_name}: {f}" for f in cross_sectional_features]
    + macro_features
)
assert len(obs_names) == 46, f"Expected 46 observation dimensions, got {len(obs_names)}"
print(f"✅ All 46 observation dimensions verified for benchmark {benchmark_name}.")

✅ All 46 observation dimensions verified for benchmark SPY.


In [ ]:
# CELL 7: High-Level OOS Performance Summary
print("=" * 60)
print(f"🏆 CHAMPION MODEL LOADED: {target_file}")
print(f"🌱 Winning Seed: {results.get('metadata', {}).get('SEEDS', 'N/A')}")
print("=" * 60)
print(f"Total Cumulative Return : {results['total_return']*100:.2f}%")
print(f"Sharpe Ratio (Ann)      : {results['sharpe_ratio']:.3f}")
print(f"Sortino Ratio (Ann)     : {results['sortino_ratio']:.3f}")
print(f"Max Drawdown            : {results['max_drawdown']*100:.2f}%")
print(f"Information Ratio       : {results['information_ratio']:.3f}")
print(f"Beta to {benchmark_name}       : {results['beta']:.2f}")
print(f"Total Trading Days      : {results['steps']} days")
print("=" * 60)

🏆 CHAMPION MODEL LOADED: model_T2_LR20_Ent005_Scale25_s42_ent0.005_mult2.5_tilt0.75_w3_g0.9_rsc25.0_lr0.0002_ep_72_rollsh_1.14_valsh_1.12.pt
🌱 Winning Seed: [42]
Total Cumulative Return : 179.79%
Sharpe Ratio (Ann)      : 0.984
Sortino Ratio (Ann)     : 1.417
Max Drawdown            : -25.20%
Information Ratio       : 0.665
Beta to SPY       : 1.21
Total Trading Days      : 1097 days


In [ ]:
# CELL 8: Strategy Personality & Allocation Dial Distribution
mean_actions = blotter_df[action_names].mean().reset_index()
mean_actions.columns = ["Action Dimension", "Average Value"]
mean_actions = mean_actions.sort_values("Average Value", ascending=True)

fig = px.bar(
    mean_actions,
    x="Average Value",
    y="Action Dimension",
    orientation="h",
    title=f"🤖 Agent Personality & Factor Tilts (OOS Average) — {target_file}",
    color="Average Value",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=650, width=950)
fig.show()

In [ ]:
# CELL 9: 4-Row Performance & Allocation Timeline
blotter_df["date"] = pd.to_datetime(blotter_df["date"])
blotter_df["Benchmark_Equity"] = (1.0 + blotter_df["bm_daily_simple_ret"]).cumprod()
blotter_df["Peak"] = blotter_df["agent_equity"].cummax()
blotter_df["Drawdown"] = (blotter_df["agent_equity"] / blotter_df["Peak"]) - 1.0

t0_start_date = blotter_df["date"].iloc[0]
t0_row = pd.DataFrame(
    [
        {
            "plot_date": t0_start_date,
            "agent_equity": 1.0,
            "Benchmark_Equity": 1.0,
            "alpha_equity": 1.0,
            "Drawdown": 0.0,
            "weight_active": blotter_df["weight_active"].iloc[0],
            "weight_benchmark": blotter_df["weight_benchmark"].iloc[0],
            "weight_cash": blotter_df["weight_cash"].iloc[0],
        }
    ]
)

realized_series = pd.DataFrame(
    {
        "plot_date": blotter_df["date"],
        "agent_equity": blotter_df["agent_equity"],
        "Benchmark_Equity": blotter_df["Benchmark_Equity"],
        "alpha_equity": blotter_df["alpha_equity"],
        "Drawdown": blotter_df["Drawdown"],
        "weight_active": blotter_df["weight_active"],
        "weight_benchmark": blotter_df["weight_benchmark"],
        "weight_cash": blotter_df["weight_cash"],
    }
)

plot_df = pd.concat([t0_row, realized_series], ignore_index=True)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.38, 0.22, 0.20, 0.20],
    subplot_titles=(
        f"📈 OOS Realized Equity Curve (Agent vs {benchmark_name})",
        "🔥 Cumulative Alpha Equity (Pure Spread Outperformance)",
        f"🏛️ Dynamic Tri-Asset Allocation (Active Basket / {benchmark_name} / Cash)",
        "📉 Drawdown Profile",
    ),
)

fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["agent_equity"],
        mode="lines",
        name="Agent Strategy",
        line=dict(color="blue", width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["Benchmark_Equity"],
        mode="lines",
        name=f"Benchmark ({benchmark_name})",
        line=dict(color="gray", width=2, dash="dash"),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["alpha_equity"],
        mode="lines",
        name="Alpha Equity",
        line=dict(color="purple", width=2),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=1.0, line_dash="dot", line_color="black", row=2, col=1)

fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["weight_active"],
        mode="lines",
        stackgroup="alloc",
        name="Active Stocks",
        line=dict(color="forestgreen", width=0.5),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["weight_benchmark"],
        mode="lines",
        stackgroup="alloc",
        name=f"Benchmark ({benchmark_name})",
        line=dict(color="royalblue", width=0.5),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["weight_cash"],
        mode="lines",
        stackgroup="alloc",
        name="Cash",
        line=dict(color="gold", width=0.5),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=plot_df["plot_date"],
        y=plot_df["Drawdown"],
        mode="lines",
        fill="tozeroy",
        name="Drawdown",
        line=dict(color="red", width=1),
    ),
    row=4,
    col=1,
)

fig.update_layout(
    height=1100,
    width=1100,
    hovermode="x unified",
    title_text=f"Performance & Allocation Profile ({target_file})",
)
fig.update_yaxes(title_text="Multiplier", row=1, col=1)
fig.update_yaxes(title_text="Alpha Multiplier", row=2, col=1)
fig.update_yaxes(
    title_text="Exposure %", tickformat=".0%", range=[0, 1.0], row=3, col=1
)
fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", row=4, col=1)
fig.show()

In [ ]:
# CELL 10: Complete Universe Selection & Trade Blotter Debug Inspector
sample_idx = -1
row = blotter_df.iloc[sample_idx]

u_size = row.get("universe_size", "N/A")
r_offset = row.get("offset", 0)
r_width = row.get("width", 0)
selected_tickers = row.get("tickers", [])
top_3 = row.get("top_3", [])

w_act = row.get("weight_active", 1.0)
w_bm = row.get("weight_benchmark", 0.0)
w_csh = row.get("weight_cash", 0.0)
eq_exp = row.get("equity_exposure", 1.0)
act_tilt = row.get("active_tilt", 1.0)


def fmt_date(d):
    return (
        pd.to_datetime(d).strftime("%Y-%m-%d")
        if pd.notna(d) and d is not None
        else "N/A"
    )


date_str = fmt_date(row.get("date"))
buy_str = fmt_date(row.get("buy_date"))
sell_str = fmt_date(row.get("sell_date"))

print("=" * 75)
print(f"📋 UNIVERSE SELECTION & DEBUG INSPECTOR — Decision Date: {date_str}")
print("=" * 75)

# 1. Timeline
print("🗓️  1. TRADE TIMELINE")
print(f"  • Decision Date (T)     : {date_str}")
print(f"  • Execution Buy Date    : {buy_str}")
print(f"  • Realized Sell Date    : {sell_str}")
print(f"  • Holding Period Buffer : {config.holding_period} trading days")
print("-" * 75)

# 2. Capital Allocation
print("🏛️  2. CAPITAL ALLOCATION & EXECUTION DIALS")
print(
    f"  • Total Equity Exposure : {eq_exp*100:.2f}% (Remainder {100 - eq_exp*100:.2f}% in Cash)"
    if isinstance(eq_exp, (int, float))
    else f"  • Total Equity Exposure : {eq_exp}"
)
print(
    f"  • Active Stock Tilt     : {act_tilt*100:.2f}% (Active Stocks vs Benchmark)"
    if isinstance(act_tilt, (int, float))
    else f"  • Active Stock Tilt     : {act_tilt}"
)
print(f"  • Effective w_active    : {w_act*100:.2f}% (Single-Stock Active Basket)")
print(
    f"  • Effective w_benchmark : {w_bm*100:.2f}% (Benchmark {config.benchmark_ticker})"
)
print(f"  • Effective w_cash      : {w_csh*100:.2f}% (Risk-Free Cash)")
print("-" * 75)

# 3. Strategy Factor Weights
print("🧠  3. ACTIVE FACTOR WEIGHTS (Cross-Sectional Scoring)")
strat_weights = []
for strat_name in action_names:
    if strat_name not in [
        "Rank Offset",
        "Rank Width",
        "Equity Exposure",
        "Active Tilt",
    ]:
        if strat_name in row:
            strat_weights.append((strat_name, row[strat_name]))

if strat_weights:
    for name, wt in sorted(strat_weights, key=lambda x: abs(x[1]), reverse=True):
        bars_count = int(abs(wt) * 25)
        bar_char = "+" if wt >= 0 else "-"
        bar_str = bar_char * bars_count
        print(f"  • {name:<26} : {wt:+.4f} | [{bar_str:<25}]")
print("-" * 75)

# 4. Universe Ranking & Scores
print("🎯  4. UNIVERSE SELECTION & SCORES")
print(f"  • Eligible Universe Size : {u_size} tickers")
print(f"  • Cross-Section Max Score: {row.get('max_score', 0.0):+.4f}")
print(f"  • Cross-Section Min Score: {row.get('min_score', 0.0):+.4f}")
offset_val = (
    int(r_offset)
    if pd.notna(r_offset) and isinstance(r_offset, (int, float, np.integer))
    else 0
)
print(f"  • Decoded Rank Offset    : {offset_val} (Starts at Rank {offset_val + 1})")
print(f"  • Decoded Rank Width     : {r_width} tickers")
print(f"  • Top 3 Overall Universe : {top_3}")
print("-" * 75)

# 5. Selected Tickers
print(f"📦  5. ALL {len(selected_tickers)} SELECTED TICKERS:")
if len(selected_tickers) > 0:
    for rank_i, ticker in enumerate(selected_tickers, start=1):
        print(
            f"    [{rank_i:02d}] Ticker: {ticker:<6} | Universe Rank: {offset_val + rank_i} of {u_size}"
        )
else:
    print(
        f"    [NONE] 100% Capital allocated between Benchmark ({config.benchmark_ticker}) and Cash."
    )
print("-" * 75)

# 6. Financials & MTM Return Accounting
print("💰  6. DAILY MTM RETURN & ALPHA ACCOUNTING")
r_stock = row.get("gross_stock_daily_simple_ret", 0.0)
r_mkt = row.get("bm_daily_simple_ret", 0.0)
r_net = row.get("net_daily_simple_ret", 0.0)
r_alpha = row.get("alpha_daily_simple_ret", 0.0)
slippage = row.get("slippage_daily_simple_loss", 0.0)

print(f"  • Daily Active Stock Return: {r_stock*100:+.3f}%")
print(f"  • Daily Benchmark Return ({config.benchmark_ticker}): {r_mkt*100:+.3f}%")
print(f"  • Daily Amortized Slippage : -{slippage*100:.4f}% ({slippage*10000:.1f} bps)")
print(f"  • Daily Net Portfolio Return: {r_net*100:+.3f}%")
print(f"  • Daily Pure Alpha Spread  : {r_alpha*100:+.3f}% (Net of Slippage & Beta)")
print(f"  • Daily Agent Equity       : {row.get('agent_equity', 1.0):.4f}")
print(f"  • Cumulative Alpha Equity  : {row.get('alpha_equity', 1.0):.4f}")
print("=" * 75)

# 7. Raw Blotter Record Dump (Deep Debugging)
print("🔍  7. COMPLETE RAW BLOTTER ROW RECORD:")
with pd.option_context("display.max_rows", None, "display.width", 1000):
    for k, v in row.items():
        if k == "observation":
            obs_sample = np.array(v).flatten()
            print(
                f"  • {k:<24} : Array shape ({obs_sample.shape[0]},) | Min: {obs_sample.min():.3f}, Max: {obs_sample.max():.3f}"
            )
        elif k == "raw_actions":
            act_sample = np.array(v).flatten()
            print(
                f"  • {k:<24} : Array shape ({act_sample.shape[0]},) | Values: {[round(float(x), 4) for x in act_sample]}"
            )
        else:
            print(f"  • {k:<24} : {v}")
print("=" * 75)

📋 UNIVERSE SELECTION & DEBUG INSPECTOR — Decision Date: 2026-08-17
🗓️  1. TRADE TIMELINE
  • Decision Date (T)     : 2026-08-17
  • Execution Buy Date    : 2026-08-18
  • Realized Sell Date    : 2026-08-19
  • Holding Period Buffer : 5 trading days
---------------------------------------------------------------------------
🏛️  2. CAPITAL ALLOCATION & EXECUTION DIALS
  • Total Equity Exposure : 100.00% (Remainder 0.00% in Cash)
  • Active Stock Tilt     : 88.49% (Active Stocks vs Benchmark)
  • Effective w_active    : 88.49% (Single-Stock Active Basket)
  • Effective w_benchmark : 11.51% (Benchmark SPY)
  • Effective w_cash      : 0.00% (Risk-Free Cash)
---------------------------------------------------------------------------
🧠  3. ACTIVE FACTOR WEIGHTS (Cross-Sectional Scoring)
  • Log Price Gain             : -0.6649 | [----------------         ]
  • Slope_P_5_Z                : -0.4129 | [----------               ]
  • Return Autocorr (15d)      : -0.3413 | [--------               

In [ ]:
# CELL 11: Top 5 Outperformance Days & Crash Day Radar
print("🌟 TOP 5 HIGHEST RETURN DAYS (Net of Slippage) 🌟")
top_5 = blotter_df.nlargest(5, "net_daily_simple_ret")[
    [
        "date",
        "net_daily_simple_ret",
        "bm_daily_simple_ret",
        "alpha_daily_simple_ret",
        "agent_equity",
    ]
    + action_names[:3]
].copy()
top_5["date"] = top_5["date"].dt.strftime("%Y-%m-%d")
for col in ["net_daily_simple_ret", "bm_daily_simple_ret", "alpha_daily_simple_ret"]:
    top_5[col] = top_5[col].apply(lambda x: f"{x:.4%}")
top_5["agent_equity"] = top_5["agent_equity"].apply(lambda x: f"{x:.3f}")
for col in action_names[:3]:
    top_5[col] = top_5[col].apply(lambda x: f"{float(x):.4f}")

print(top_5.to_string(index=False))

print("\n" + "=" * 50)
print("🚨 CRASH DAY ANALYSIS (Worst Single-Day Return) 🚨")
worst_idx = blotter_df["net_daily_simple_ret"].idxmin()
worst_day = blotter_df.iloc[worst_idx]
day_before = blotter_df.iloc[max(0, worst_idx - 1)]

print(
    f"Worst Day: {worst_day['date'].strftime('%Y-%m-%d')} | "
    f"Net Return: {worst_day['net_daily_simple_ret']*100:.2f}% | "
    f"Benchmark: {worst_day['bm_daily_simple_ret']*100:.2f}% | "
    f"Alpha: {worst_day['alpha_daily_simple_ret']*100:.2f}% | "
    f"Portfolio Multiplier: {worst_day['agent_equity']:.3f}"
)
print("-" * 50)

crash_weights = pd.DataFrame(
    {
        "Dimension": action_names,
        "Weight on Worst Day": worst_day[action_names].values,
        "Weight Day Prior": day_before[action_names].values,
    }
)

fig = px.line_polar(
    crash_weights,
    r="Weight on Worst Day",
    theta="Dimension",
    line_close=True,
    title=f"Radar: Agent Factor Allocation on {worst_day['date'].strftime('%Y-%m-%d')} (Crash Day)",
)
fig.update_traces(fill="toself", line_color="red")
fig.show()

🌟 TOP 5 HIGHEST RETURN DAYS (Net of Slippage) 🌟
      date net_daily_simple_ret bm_daily_simple_ret alpha_daily_simple_ret agent_equity Log Price Gain Sharpe (TRP) Momentum (21d)
2022-11-09             15.9286%             5.4953%               10.4333%        1.084        -0.6296      -0.5019        -0.3366
2022-05-12             10.9117%             2.3907%                8.5210%        0.920        -0.7052      -0.2616        -0.1699
2025-04-08             10.8944%            10.5018%                0.3926%        1.748        -0.2364      -0.5420        -0.2005
2022-11-29              9.7978%             3.1500%                6.6477%        1.269        -0.3446       0.2508         0.3734
2022-10-24              8.6362%             1.5970%                7.0392%        0.953        -0.6470      -0.2764        -0.3006

🚨 CRASH DAY ANALYSIS (Worst Single-Day Return) 🚨
Worst Day: 2022-10-18 | Net Return: -7.82% | Benchmark: -0.71% | Alpha: -7.11% | Portfolio Multiplier: 0.902
-------

In [14]:
# CELL 12: Observation Space Sanity & Outlier Bounds (46 Dimensions)
obs_raw = blotter_df["observation"].values
obs_array = np.vstack([np.array(x).flatten() for x in obs_raw])
assert (
    obs_array.shape[1] == 46
), f"Expected 46 observation dimensions, got {obs_array.shape[1]}"

obs_df = pd.DataFrame(obs_array, columns=obs_names, index=blotter_df["date"])
desc_stats = obs_df.describe().T[["mean", "std", "min", "max"]]


def format_with_clip(val: float) -> str:
    if pd.isna(val):
        return ""
    formatted = f"{val:.4f}"
    if abs(abs(val) - 4.0) < 1e-3:
        return f"{formatted} [CLIP]"
    return formatted


mapping_table = pd.DataFrame(
    {
        "Dim Index": [f"Dim_{i:02d}" for i in range(46)],
        "Feature Name": obs_names,
        "Mean": desc_stats["mean"].map(lambda x: f"{x:.4f}").values,
        "Std Dev": desc_stats["std"].map(lambda x: f"{x:.4f}").values,
        "Min": desc_stats["min"].map(format_with_clip).values,
        "Max": desc_stats["max"].map(format_with_clip).values,
    }
)

print("\n--- Mapped Observation Space Statistics (46 Dimensions) ---")
print(mapping_table.to_string(index=False))

fig = go.Figure()
for col in obs_df.columns:
    fig.add_trace(
        go.Box(
            y=obs_df[col],
            name=col,
            boxpoints="outliers",
            jitter=0.5,
            marker=dict(size=2),
        )
    )

fig.update_layout(
    title=f"🤖 Agent Observation Space Distribution (46 Dimensions) — {target_file}",
    yaxis_title="Normalized Value",
    xaxis_title="Observation Dimension",
    height=650,
    width=1250,
    showlegend=False,
    shapes=[
        dict(
            type="line",
            y0=3.0,
            y1=3.0,
            x0=-0.5,
            x1=45.5,
            line=dict(color="orange", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=-3.0,
            y1=-3.0,
            x0=-0.5,
            x1=45.5,
            line=dict(color="orange", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=4.0,
            y1=4.0,
            x0=-0.5,
            x1=45.5,
            line=dict(color="red", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=-4.0,
            y1=-4.0,
            x0=-0.5,
            x1=45.5,
            line=dict(color="red", width=1, dash="dash"),
        ),
    ],
)
fig.show()


--- Mapped Observation Space Statistics (46 Dimensions) ---
Dim Index                     Feature Name    Mean Std Dev     Min    Max
   Dim_00         Mean: Log Price Gain (Z)  0.0467  0.5907 -1.6433 1.4980
   Dim_01           Mean: Sharpe (TRP) (S)  0.0516  0.7558 -1.8505 1.9477
   Dim_02         Mean: Momentum (21d) (S)  0.0234  0.6696 -1.6770 1.6349
   Dim_03       Mean: Info Ratio (63d) (S) -0.8048  0.8149 -2.2644 1.4523
   Dim_04        Mean: Oversold (-RSI) (S) -0.0434  0.7507 -1.6642 1.9233
   Dim_05     Mean: Dip Buyer (-dd_21) (S) -0.0895  0.5719 -0.9471 1.7426
   Dim_06   Mean: Range Position (20d) (S) -0.0216  0.7856 -1.7927 1.3871
   Dim_07  Mean: Return Autocorr (15d) (S)  0.0120  0.7124 -2.1402 1.9888
   Dim_08 Mean: Low Volatility (-ATRP) (Z)  0.2224  0.3640 -0.6218 0.6791
   Dim_09            Mean: Slope_P_5_Z (Z)  0.0241  0.7215 -2.2943 1.8494
   Dim_10            Mean: Slope_V_5_Z (Z)  0.0162  0.7607 -2.1533 1.8566
   Dim_11              Mean: Convexity (Z) -0.0003 